# Spaceship Titanic — XGBoost Baseline

Kaggle's Spaceship Titanic is the modern starter competition (the successor to the original Titanic getting-started one). Binary target `Transported`, mixed feature types, lots of room for feature engineering on the `Cabin` and `PassengerId` strings.

I don't have the actual train.csv pulled, so this notebook synthesizes the schema. The structural choices (CryoSleep → low spending, group id from PassengerId, deck/side from Cabin) follow what the real dataset has. Doing it this way means the *engineering* in the notebook is correct even though the *numbers* are fabricated.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print("xgboost not installed — falling back to sklearn GradientBoostingClassifier")

rng = np.random.RandomState(13)
DATA = Path("../../data/spaceship_titanic")
DATA.mkdir(parents=True, exist_ok=True)

## Synthesize the schema

Real columns: `PassengerId, HomePlanet, CryoSleep, Cabin, Destination, Age, VIP, RoomService, FoodCourt, ShoppingMall, Spa, VRDeck, Transported`. Cabin is `<deck>/<num>/<side>`. PassengerId is `<gggg>_<pp>` (group / person-in-group).

In [2]:
N = 4000
home = rng.choice(["Earth", "Europa", "Mars"], size=N, p=[0.5, 0.25, 0.25])
cryo = rng.binomial(1, 0.3, N).astype(bool)
dest = rng.choice(["TRAPPIST-1e", "55 Cancri e", "PSO J318.5-22"], size=N, p=[0.6, 0.25, 0.15])
age = rng.normal(28, 14, N).clip(0, 80)
vip = rng.binomial(1, 0.05, N).astype(bool)
deck = rng.choice(list("ABCDEFG"), size=N, p=[0.05, 0.05, 0.15, 0.20, 0.25, 0.20, 0.10])
side = rng.choice(["P", "S"], size=N)
cabin_num = rng.randint(0, 1500, N)
group = rng.randint(1, 1500, N)
person = rng.randint(1, 6, N)

# spending: zero if cryo, otherwise lognormal per-amenity
def spend(scale, mu=2.5, sigma=1.1):
    s = np.exp(rng.normal(mu, sigma, N)) * scale
    s[cryo] = 0
    return s.round(1)

room = spend(1.0)
food = spend(1.5)
mall = spend(0.8)
spa  = spend(0.9)
vrd  = spend(0.7)

# target: cryo passengers are much more likely to be transported (this is the real-data signal too)
logit = (
    -0.2
    + 1.6 * cryo
    + 0.3 * (home == "Europa")
    - 0.2 * (home == "Mars")
    - 0.4 * (dest == "55 Cancri e")
    - 0.012 * (age - 28)
    + 0.4 * vip
    + 0.25 * (deck == "B").astype(int)
    - 0.000003 * (room + food + mall + spa + vrd)
    + rng.normal(0, 0.5, N)
)
p_trans = 1 / (1 + np.exp(-logit))
transported = (rng.uniform(size=N) < p_trans)

df = pd.DataFrame({
    "PassengerId": [f"{g:04d}_{p:02d}" for g, p in zip(group, person)],
    "HomePlanet": home,
    "CryoSleep": cryo,
    "Cabin": [f"{d}/{n}/{s}" for d, n, s in zip(deck, cabin_num, side)],
    "Destination": dest,
    "Age": age.round(1),
    "VIP": vip,
    "RoomService": room, "FoodCourt": food, "ShoppingMall": mall, "Spa": spa, "VRDeck": vrd,
    "Transported": transported,
})

# inject a few NaNs like the real dataset has — mostly in Cabin and Age
for col in ["Cabin", "Age", "HomePlanet"]:
    mask = rng.rand(N) < 0.02
    df.loc[mask, col] = np.nan

df.to_csv(DATA / "train_synth.csv", index=False)
print(df.shape, "  target rate:", df.Transported.mean().round(3))
df.head()

(4000, 13)   target rate: 0.548


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,0354_03,Mars,False,D/251/P,55 Cancri e,37.3,False,8.1,6.2,2.9,13.5,3.8,True
1,1039_03,Earth,False,E/220/P,PSO J318.5-22,42.9,False,10.1,5.1,8.0,14.3,27.5,True
2,0553_03,Mars,False,C/260/P,TRAPPIST-1e,9.1,False,34.2,8.9,3.7,8.8,8.2,False
3,0296_04,Mars,False,G/355/S,TRAPPIST-1e,0.0,False,88.6,20.8,8.8,14.0,12.7,True
4,0796_04,Mars,True,E/1243/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,True


## Feature engineering

The bits that actually moved the leaderboard on the real competition:
- Split `Cabin` into deck/num/side.
- Group size from `PassengerId` prefix.
- Total spend = sum of amenity columns; spend==0 is a strong CryoSleep proxy.

In [3]:
def engineer(df):
    df = df.copy()
    df["GroupId"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("GroupId")["GroupId"].transform("count")

    cab = df["Cabin"].fillna("U/0/U").str.split("/", expand=True)
    df["Deck"], df["CabinNum"], df["Side"] = cab[0], cab[1].astype(float, errors="ignore"), cab[2]

    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df[spend_cols] = df[spend_cols].fillna(0)
    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["ZeroSpend"] = (df["TotalSpend"] == 0).astype(int)

    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["HomePlanet"] = df["HomePlanet"].fillna("Earth")

    # cast booleans to int and one-hot the categoricals
    df["CryoSleep"] = df["CryoSleep"].astype(int)
    df["VIP"] = df["VIP"].astype(int)
    df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Deck", "Side"], drop_first=True)

    drop = ["PassengerId", "Cabin", "GroupId"]
    return df.drop(columns=[c for c in drop if c in df.columns])

df_fe = engineer(df)
X = df_fe.drop(columns=["Transported"]).values
y = df_fe["Transported"].astype(int).values
print("features:", df_fe.shape[1] - 1)

features: 25


## Train XGBoost (or fall back to sklearn GB)

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=13)

if HAS_XGB:
    model = XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, random_state=13,
        eval_metric="logloss", use_label_encoder=False,
    )
else:
    model = GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=13)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
auc_cv = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"CV AUC: mean={auc_cv.mean():.3f}  std={auc_cv.std():.3f}")

model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
print(f"test AUC: {roc_auc_score(y_te, proba):.3f}")
print(f"test acc: {accuracy_score(y_te, pred):.3f}")
print(classification_report(y_te, pred, target_names=["not_transported", "transported"]))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


CV AUC: mean=0.650  std=0.011


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:21:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


test AUC: 0.623
test acc: 0.589
                 precision    recall  f1-score   support

not_transported       0.54      0.55      0.55       362
    transported       0.63      0.62      0.62       438

       accuracy                           0.59       800
      macro avg       0.59      0.59      0.59       800
   weighted avg       0.59      0.59      0.59       800



## Notes

What I'd port directly to the real Kaggle train.csv:
- The `engineer()` function — feature engineering is portable.
- The XGBoost hyperparameters are a reasonable starting band but obviously need real tuning (Optuna or a coarse grid).
- The CV protocol.

What's specifically *not* transferable: the AUC number above is on data I generated, so the model is fitting my generative process. Treat the numbers like a unit test, not a benchmark.

Real Spaceship Titanic public-leaderboard scores hover around 0.79-0.82 accuracy with XGBoost + decent FE — that's the band to aim for once the actual CSV is loaded.